In [1]:
import cv2
import numpy as np

In [2]:
# 1a: Cargar una imagen y mostrar el valor RGB del píxel bajo el cursor

ruta_imagen = "Imagen1.jpg"
img = cv2.imread(ruta_imagen)

if img is None:
    raise FileNotFoundError(f"No se pudo cargar la imagen en: {ruta_imagen}")

# Escalar la imagen para que quepa en pantalla sin recortarse
ANCHO_MAX, ALTO_MAX = 1280, 720
alto_original, ancho_original = img.shape[:2]

escala = min(ANCHO_MAX / ancho_original, ALTO_MAX / alto_original, 1.0)
ancho_mostrado = int(ancho_original * escala)
alto_mostrado = int(alto_original * escala)

img_mostrada = cv2.resize(img, (ancho_mostrado, alto_mostrado), interpolation=cv2.INTER_AREA)

ventana = "Practica 1a - Valor RGB bajo el cursor"
cv2.namedWindow(ventana)


def mostrar_rgb(event, x, y, flags, param):
    if event == cv2.EVENT_MOUSEMOVE:
        # Convertimos la posición del ratón (sobre la imagen escalada)
        # a coordenadas de la imagen original, para leer el píxel real
        x_original = min(int(x / escala), ancho_original - 1)
        y_original = min(int(y / escala), alto_original - 1)

        b, g, r = img[y_original, x_original]
        texto = f"RGB: ({r}, {g}, {b})  Pos: ({x_original}, {y_original})"

        vista = img_mostrada.copy()
        cv2.putText(vista, texto, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow(ventana, vista)


cv2.setMouseCallback(ventana, mostrar_rgb)
cv2.imshow(ventana, img_mostrada)

print("Mueve el ratón sobre la imagen para ver el valor RGB del píxel.")
print("Pulsa 'q' o ESC para salir.")

while True:
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()

Mueve el ratón sobre la imagen para ver el valor RGB del píxel.
Pulsa 'q' o ESC para salir.


In [ ]:
import cv2
import numpy as np
import tkinter as tk

# Resolución de pantalla (para que la interfaz se dibuje nítida, sin estirarse)
def tamano_pantalla():
    raiz = tk.Tk()
    raiz.withdraw()
    ancho, alto = raiz.winfo_screenwidth(), raiz.winfo_screenheight()
    raiz.destroy()
    return ancho, alto

PANT_ANCHO, PANT_ALTO = tamano_pantalla()

# Dimensiones y escalado
BASE_ANCHO_LIENZO, BASE_ALTO, BASE_ANCHO_PANEL = 720, 760, 230
BASE_M, BASE_ALTO_B, BASE_ESP = 14, 26, 6
BASE_RADIO_REDONDEO = 8
BASE_ALTO_SLIDER, BASE_ALTO_PREVIEW = 26, 34
BASE_F_TITULO, BASE_F_BOTON, BASE_F_PREVIEW, BASE_F_INFO = 0.62, 0.45, 0.40, 0.42

MARGEN = 80
_escala = min((PANT_ANCHO - MARGEN) / (BASE_ANCHO_PANEL + BASE_ANCHO_LIENZO),
              (PANT_ALTO - MARGEN) / BASE_ALTO)
ESCALA = max(0.6, min(_escala, 2.0))

ANCHO_LIENZO   = int(BASE_ANCHO_LIENZO * ESCALA)
ALTO           = int(BASE_ALTO * ESCALA)
ANCHO_PANEL    = int(BASE_ANCHO_PANEL * ESCALA)
M              = int(BASE_M * ESCALA)
ALTO_B         = int(BASE_ALTO_B * ESCALA)
ESP            = int(BASE_ESP * ESCALA)
RADIO_REDONDEO = max(2, int(BASE_RADIO_REDONDEO * ESCALA))
ALTO_SLIDER    = int(BASE_ALTO_SLIDER * ESCALA)
ALTO_PREVIEW   = int(BASE_ALTO_PREVIEW * ESCALA)
F_TITULO       = BASE_F_TITULO * ESCALA
F_BOTON        = BASE_F_BOTON * ESCALA
F_PREVIEW      = BASE_F_PREVIEW * ESCALA
F_INFO         = BASE_F_INFO * ESCALA
GROSOR_TEXTO   = 2 if ESCALA >= 1.35 else 1

ANCHO_VENTANA, ALTO_VENTANA = ANCHO_PANEL + ANCHO_LIENZO, ALTO
VENTANA = "PIAV - Practica 1 (tema lila)"

BG_PANEL   = (60, 30, 45)
BG_BOTON   = (120, 70, 90)
BG_HOVER   = (160, 95, 120)
BG_SEL     = (225, 120, 190)
SEPARADOR  = (120, 80, 100)
TXT        = (240, 230, 245)
TXT_SEL    = (35, 10, 30)
FUENTE = cv2.FONT_HERSHEY_SIMPLEX

# Estado
capa_libre = np.ones((ALTO, ANCHO_LIENZO, 3), dtype=np.uint8) * 255
figuras = []
historial = [(capa_libre.copy(), [])]

modo = "linea"
relleno = False
simetria = "ninguna"
cuentagotas = False
color_actual = (0, 0, 0)
color_preview = None
grosor = max(2, round(3 * ESCALA))
dibujando = False
pt_inicio = (0, 0)
puntos_poligono = []
ultimo_punto_libre = None
cambios_goma = False
grabando = False
video_writer = None
FPS = 20
hover_id = None
arrastrando_slider = False

def rect_redondeado(img, p1, p2, color, radio=RADIO_REDONDEO, grosor_b=-1):
    x1, y1 = p1; x2, y2 = p2
    if grosor_b < 0:
        cv2.rectangle(img, (x1+radio, y1), (x2-radio, y2), color, -1)
        cv2.rectangle(img, (x1, y1+radio), (x2, y2-radio), color, -1)
        for cx, cy in [(x1+radio,y1+radio),(x2-radio,y1+radio),
                        (x1+radio,y2-radio),(x2-radio,y2-radio)]:
            cv2.circle(img, (cx, cy), radio, color, -1)
    else:
        cv2.rectangle(img, (x1+radio, y1), (x2-radio, y2), color, grosor_b)
        cv2.rectangle(img, (x1, y1+radio), (x2, y2-radio), color, grosor_b)

botones = []
def boton(id_b, texto, x1, y1, x2, y2):
    botones.append({"id": id_b, "texto": texto, "rect": (x1, y1, x2, y2)})

AB = ANCHO_PANEL - 2 * M
y = int(64 * ESCALA)
for id_h, txt in [("linea","Linea"), ("rectangulo","Rectangulo"),
                   ("circulo","Circulo"), ("elipse","Elipse"),
                   ("polilinea","Polilinea (abierta)"),
                   ("poligono","Poligono (cerrado)"),
                   ("libre","Trazo libre"), ("goma","Goma de borrar")]:
    boton(id_h, txt, M, y, M+AB, y+ALTO_B); y += ALTO_B+ESP

y += int(8 * ESCALA)
for id_t in ["relleno", "simetria", "cuentagotas"]:
    boton(id_t, "", M, y, M+AB, y+ALTO_B); y += ALTO_B+ESP

y += int(12 * ESCALA)
boton("grosor_menos", "-", M, y, M+AB//2-4, y+ALTO_B)
boton("grosor_mas", "+", M+AB//2+4, y, M+AB, y+ALTO_B)
y += ALTO_B+ESP+int(22 * ESCALA)

SLIDER_RECT = (M, y, M + AB, y + ALTO_SLIDER)
_ancho_s = AB
alto_mitad_hsv = ALTO_SLIDER // 2
alto_mitad_gris = ALTO_SLIDER - alto_mitad_hsv

# Mitad superior: Espectro de color HSV (brillante)
_tira_hsv = np.zeros((1, _ancho_s, 3), dtype=np.uint8)
for i in range(_ancho_s):
    _tira_hsv[0, i] = (int(179 * i / (_ancho_s - 1)), 255, 255)
tira_colores = np.repeat(cv2.cvtColor(_tira_hsv, cv2.COLOR_HSV2BGR), alto_mitad_hsv, axis=0)

# Mitad inferior: Escala de grises (de Negro a Blanco)
tira_grises = np.zeros((alto_mitad_gris, _ancho_s, 3), dtype=np.uint8)
for i in range(_ancho_s):
    val_gris = int(255 * i / (_ancho_s - 1))
    tira_grises[:, i] = (val_gris, val_gris, val_gris)

TIRA_COLOR = np.vstack((tira_colores, tira_grises))
y += ALTO_SLIDER + int(10 * ESCALA)

PREVIEW_RECT = (M, y, M+AB, y+ALTO_PREVIEW)
y += ALTO_PREVIEW + int(14 * ESCALA)

for id_a, txt in [("deshacer","Deshacer"), ("borrar","Borrar todo"),
                   ("guardar","Guardar imagen"), ("grabar","")]:
    boton(id_a, txt, M, y, M+AB, y+ALTO_B); y += ALTO_B+ESP

def etiqueta(id_b):
    if id_b == "relleno":     return f"Relleno: {'SI' if relleno else 'NO'}"
    if id_b == "simetria":    return f"Simetria: {simetria.capitalize()}"
    if id_b == "cuentagotas": return f"Pipeta color: {'SI' if cuentagotas else 'NO'}"
    if id_b == "grabar":      return "Detener video" if grabando else "Grabar video"
    return next(b["texto"] for b in botones if b["id"] == id_b)

def seleccionado(id_b):
    return (id_b == modo or (id_b=="relleno" and relleno) or
            (id_b=="cuentagotas" and cuentagotas) or (id_b=="grabar" and grabando))

def dibujar_boton(panel, b):
    x1, y1, x2, y2 = b["rect"]; id_b = b["id"]
    if seleccionado(id_b):   fondo, tcol = BG_SEL, TXT_SEL
    elif id_b == hover_id:   fondo, tcol = BG_HOVER, TXT
    else:                    fondo, tcol = BG_BOTON, TXT
    rect_redondeado(panel, (x1,y1), (x2,y2), fondo)
    texto = etiqueta(id_b)
    (tw, th), _ = cv2.getTextSize(texto, FUENTE, F_BOTON, GROSOR_TEXTO)
    cv2.putText(panel, texto, (x1+(x2-x1-tw)//2, y1+(y2-y1+th)//2),
                FUENTE, F_BOTON, tcol, GROSOR_TEXTO, cv2.LINE_AA)

def dibujar_slider(panel):
    x1, y1, x2, y2 = SLIDER_RECT
    panel[y1:y2, x1:x2] = TIRA_COLOR
    cv2.rectangle(panel, (x1, y1), (x2, y2), (200, 180, 210), 1)
    alto_mitad = ALTO_SLIDER // 2
    cv2.line(panel, (x1, y1 + alto_mitad), (x2, y1 + alto_mitad), (100, 100, 100), 1)

    b, g, r = color_actual
    if b == g == r:
        pos_x = x1 + int((b / 255.0) * (x2 - x1 - 1))
        sy1, sy2 = y1 + alto_mitad, y2
    else:
        hsv = cv2.cvtColor(np.uint8([[color_actual]]), cv2.COLOR_BGR2HSV)[0][0]
        pos_x = x1 + int((hsv[0] / 179.0) * (x2 - x1 - 1))
        sy1, sy2 = y1, y1 + alto_mitad

    cv2.line(panel, (pos_x, sy1), (pos_x, sy2), (255, 255, 255), 2)
    cv2.line(panel, (pos_x, sy1), (pos_x, sy2), (30, 10, 25), 1)

def dibujar_preview(panel):
    x1, y1, x2, y2 = PREVIEW_RECT
    mostrado = color_preview if color_preview is not None else color_actual
    rect_redondeado(panel, (x1,y1), (x2,y2), mostrado)
    cv2.rectangle(panel, (x1,y1), (x2,y2), (200,180,210), 1)
    etiq = "Previsualizando..." if color_preview is not None else "Color actual"
    b,g,r = mostrado
    col_texto = (255,255,255) if (r+g+b) < 380 else (20,10,15)
    cv2.putText(panel, f"{etiq} RGB({r},{g},{b})", (x1+8, y1+int(22*ESCALA)),
                FUENTE, F_PREVIEW, col_texto, GROSOR_TEXTO, cv2.LINE_AA)

def construir_panel():
    panel = np.full((ALTO, ANCHO_PANEL, 3), BG_PANEL, dtype=np.uint8)
    cv2.putText(panel, "PIAV - Dibujo", (M, int(34*ESCALA)), FUENTE, F_TITULO,
                (235,210,240), GROSOR_TEXTO, cv2.LINE_AA)
    cv2.line(panel, (0,int(48*ESCALA)), (ANCHO_PANEL,int(48*ESCALA)), SEPARADOR, 1)
    for b in botones:
        dibujar_boton(panel, b)
    dibujar_slider(panel)
    dibujar_preview(panel)
    gx1, gy1, gx2, gy2 = next(b["rect"] for b in botones if b["id"]=="grosor_mas")
    cv2.putText(panel, f"Grosor/Goma: {grosor}px", (M, gy2+int(20*ESCALA)),
                FUENTE, F_INFO, TXT, GROSOR_TEXTO, cv2.LINE_AA)
    return panel

def variantes_punto(p):
    x, y = p
    v = [p]
    if simetria in ("horizontal", "ambas"): v.append((ANCHO_LIENZO - x, y))
    if simetria in ("vertical", "ambas"):   v.append((x, ALTO - y))
    if simetria == "ambas":                 v.append((ANCHO_LIENZO - x, ALTO - y))
    return v

def trazar(destino, tipo, datos, color, gro):
    if tipo == "linea":
        for p1, p2 in datos: cv2.line(destino, p1, p2, color, max(gro,1))
    elif tipo == "rectangulo":
        for p1, p2 in datos: cv2.rectangle(destino, p1, p2, color, gro)
    elif tipo == "circulo":
        for c, r in datos: cv2.circle(destino, c, r, color, gro)
    elif tipo == "elipse":
        for c, ejes in datos: cv2.ellipse(destino, c, ejes, 0, 0, 360, color, gro)
    elif tipo == "poligono":
        for pts in datos:
            arr = np.array(pts, np.int32).reshape((-1,1,2))
            if gro == -1: cv2.fillPoly(destino, [arr], color)
            else: cv2.polylines(destino, [arr], True, color, max(gro,1))
    elif tipo == "polilinea":
        for pts in datos:
            arr = np.array(pts, np.int32).reshape((-1,1,2))
            cv2.polylines(destino, [arr], False, color, max(gro,1))

def renderizar():
    canvas = capa_libre.copy()
    for fig in figuras:
        gro = -1 if fig["relleno"] else fig["grosor"]
        trazar(canvas, fig["tipo"], fig["datos"], fig["color"], gro)
    return canvas

def mascara_de_figura(fig):
    mask = np.zeros((ALTO, ANCHO_LIENZO), dtype=np.uint8)
    gro = -1 if fig["relleno"] else fig["grosor"] + 10
    trazar(mask, fig["tipo"], fig["datos"], 255, gro)
    return mask

def quitar_figura_en(x, y):
    for fig in reversed(figuras):
        if mascara_de_figura(fig)[y, x] > 0:
            figuras.remove(fig)
            return True
    return False

def finalizar_trazado_multipunto():
    global puntos_poligono
    if len(puntos_poligono) >= 2:
        n = len(variantes_punto(puntos_poligono[0]))
        grupos = [[] for _ in range(n)]
        for p in puntos_poligono:
            for i, vp in enumerate(variantes_punto(p)):
                grupos[i].append(vp)
        figuras.append({"tipo": modo, "color": color_actual,
                         "grosor": max(grosor,1),
                         "relleno": relleno if modo == "poligono" else False,
                         "datos": grupos})
        registrar_paso()
    puntos_poligono = []

def trazo_libre(destino, p1, p2, color, radio):
    dist = max(int(np.hypot(p2[0]-p1[0], p2[1]-p1[1])), 1)
    for i in range(dist + 1):
        t = i / dist
        x = int(p1[0] + (p2[0]-p1[0]) * t)
        y = int(p1[1] + (p2[1]-p1[1]) * t)
        cv2.circle(destino, (x, y), radio, color, -1)

def registrar_paso():
    historial.append((capa_libre.copy(), list(figuras)))
    if grabando and video_writer is not None:
        frame = renderizar()
        for _ in range(8):
            video_writer.write(frame)

def click_en_boton(x, y):
    for b in botones:
        x1,y1,x2,y2 = b["rect"]
        if x1<=x<=x2 and y1<=y<=y2: return b["id"]
    return None

def dentro_de(rect, x, y):
    x1,y1,x2,y2 = rect
    return x1<=x<=x2 and y1<=y<=y2

def color_desde_slider(x, y_pos):
    x1, y1, x2, y2 = SLIDER_RECT
    frac_x = min(max((x - x1) / (x2 - x1 - 1), 0.0), 1.0)
    alto_mitad = ALTO_SLIDER // 2

    # Si el cursor está en la mitad inferior: Escala de grises (incluye Negro)
    if (y_pos - y1) >= alto_mitad:
        val = int(255 * frac_x)
        return (val, val, val)
    else:
        # Mitad superior: Color saturado HSV
        hue = int(179 * frac_x)
        bgr = cv2.cvtColor(np.uint8([[[hue, 255, 255]]]), cv2.COLOR_HSV2BGR)[0][0]
        return (int(bgr[0]), int(bgr[1]), int(bgr[2]))

def accionar_boton(id_b):
    global modo, relleno, simetria, cuentagotas, grosor
    global capa_libre, figuras, grabando, video_writer
    if id_b in ("linea","rectangulo","circulo","elipse","polilinea","poligono","libre","goma"):
        modo = id_b; puntos_poligono.clear()
    elif id_b == "relleno": relleno = not relleno
    elif id_b == "simetria":
        orden = ["ninguna","horizontal","vertical","ambas"]
        simetria = orden[(orden.index(simetria)+1) % 4]
    elif id_b == "cuentagotas": cuentagotas = not cuentagotas
    elif id_b == "grosor_menos": grosor = max(1, grosor-1)
    elif id_b == "grosor_mas": grosor = min(40, grosor+1)
    elif id_b == "deshacer":
        if len(historial) > 1:
            historial.pop()
            capa_libre, figs = historial[-1]
            capa_libre = capa_libre.copy(); figuras[:] = figs
    elif id_b == "borrar":
        capa_libre = np.ones((ALTO, ANCHO_LIENZO, 3), dtype=np.uint8) * 255
        figuras.clear(); historial.clear(); historial.append((capa_libre.copy(), []))
    elif id_b == "guardar":
        cv2.imwrite("dibujo_final.png", renderizar()); print("Guardado: dibujo_final.png")
    elif id_b == "grabar":
        if not grabando:
            video_writer = cv2.VideoWriter("dibujo_paso_a_paso.mp4",
                            cv2.VideoWriter_fourcc(*"mp4v"), FPS, (ANCHO_LIENZO, ALTO))
            grabando = True
        else:
            grabando = False; video_writer.release(); video_writer = None
            print("Video guardado: dibujo_paso_a_paso.mp4")

def evento_raton(event, x, y, flags, param):
    global hover_id, dibujando, pt_inicio, color_actual, color_preview
    global cuentagotas, arrastrando_slider, ultimo_punto_libre, cambios_goma

    if x < ANCHO_PANEL:
        color_preview = None
        if dentro_de(SLIDER_RECT, x, y):
            hover_id = None
            color_preview = color_desde_slider(x, y)
            if event == cv2.EVENT_LBUTTONDOWN:
                arrastrando_slider = True; color_actual = color_preview
            elif event == cv2.EVENT_MOUSEMOVE and arrastrando_slider:
                color_actual = color_preview
            return
        if event == cv2.EVENT_LBUTTONUP: arrastrando_slider = False
        id_b = click_en_boton(x, y)
        if event == cv2.EVENT_MOUSEMOVE: hover_id = id_b
        elif event == cv2.EVENT_LBUTTONDOWN and id_b: accionar_boton(id_b)
        return

    arrastrando_slider = False
    hover_id = None
    xc = x - ANCHO_PANEL

    # Pipeta de color
    if cuentagotas:
        color_preview = tuple(int(v) for v in renderizar()[y, xc])
        if event == cv2.EVENT_LBUTTONDOWN:
            color_actual = color_preview
            cuentagotas = False
            color_preview = None
            dibujando = False
        return
    color_preview = None

    if modo in ("poligono", "polilinea"):
        if event == cv2.EVENT_LBUTTONDOWN: puntos_poligono.append((xc, y))
        elif event in (cv2.EVENT_RBUTTONDOWN, cv2.EVENT_LBUTTONDBLCLK): finalizar_trazado_multipunto()
        return

    if modo == "libre":
        if event == cv2.EVENT_LBUTTONDOWN:
            dibujando = True; ultimo_punto_libre = (xc, y)
            cv2.circle(capa_libre, (xc, y), max(grosor//2, 1), color_actual, -1)
        elif event == cv2.EVENT_MOUSEMOVE and dibujando:
            trazo_libre(capa_libre, ultimo_punto_libre, (xc, y), color_actual, max(grosor//2, 1))
            ultimo_punto_libre = (xc, y)
        elif event == cv2.EVENT_LBUTTONUP:
            dibujando = False; registrar_paso()
        return

    if modo == "goma":
        radio_goma = max(grosor*2, 10)
        if event == cv2.EVENT_LBUTTONDOWN:
            dibujando = True; cambios_goma = False
            if quitar_figura_en(xc, y): cambios_goma = True
            else:
                cv2.circle(capa_libre, (xc, y), radio_goma, (255,255,255), -1)
                cambios_goma = True
            ultimo_punto_libre = (xc, y)
        elif event == cv2.EVENT_MOUSEMOVE and dibujando:
            if quitar_figura_en(xc, y):
                cambios_goma = True
            else:
                trazo_libre(capa_libre, ultimo_punto_libre, (xc, y), (255,255,255), radio_goma)
                cambios_goma = True
            ultimo_punto_libre = (xc, y)
        elif event == cv2.EVENT_LBUTTONUP:
            dibujando = False
            if cambios_goma: registrar_paso()
        return

    # Modos: linea, rectangulo, circulo, elipse
    if event == cv2.EVENT_LBUTTONDOWN:
        dibujando = True; pt_inicio = (xc, y)
    elif event == cv2.EVENT_LBUTTONUP and dibujando:
        dibujando = False
        p2 = (xc, y)
        if modo in ("linea", "rectangulo"):
            datos = list(zip(variantes_punto(pt_inicio), variantes_punto(p2)))
        elif modo == "circulo":
            radio = int(np.hypot(p2[0]-pt_inicio[0], p2[1]-pt_inicio[1]))
            datos = [(c, radio) for c in variantes_punto(pt_inicio)]
        elif modo == "elipse":
            ejes = (max(1,abs(p2[0]-pt_inicio[0])), max(1,abs(p2[1]-pt_inicio[1])))
            datos = [(c, ejes) for c in variantes_punto(pt_inicio)]
        figuras.append({"tipo":modo,"color":color_actual,"grosor":grosor,
                         "relleno":relleno,"datos":datos})
        registrar_paso()

def vista_con_preview(x, y):
    canvas = renderizar()
    if modo in ("poligono","polilinea") and puntos_poligono:
        for p in puntos_poligono: cv2.circle(canvas, p, 3, (0,0,255), -1)
        if len(puntos_poligono) > 1:
            cv2.polylines(canvas, [np.array(puntos_poligono)], False, color_actual, max(grosor,1))
        if x is not None:
            cv2.line(canvas, puntos_poligono[-1], (x - ANCHO_PANEL, y), color_actual, 1)
    elif dibujando and modo in ("linea","rectangulo","circulo","elipse") and x is not None:
        xc = x - ANCHO_PANEL; p2 = (xc, y); gro = -1 if relleno else grosor
        if modo in ("linea","rectangulo"):
            trazar(canvas, modo, list(zip(variantes_punto(pt_inicio), variantes_punto(p2))), color_actual, gro)
        elif modo == "circulo":
            radio = int(np.hypot(p2[0]-pt_inicio[0], p2[1]-pt_inicio[1]))
            trazar(canvas, "circulo", [(c, radio) for c in variantes_punto(pt_inicio)], color_actual, gro)
        elif modo == "elipse":
            ejes = (max(1,abs(p2[0]-pt_inicio[0])), max(1,abs(p2[1]-pt_inicio[1])))
            trazar(canvas, "elipse", [(c, ejes) for c in variantes_punto(pt_inicio)], color_actual, gro)
    return canvas

_ultima_pos = [None, None]
def registrar_pos(event, x, y, flags, param):
    _ultima_pos[0], _ultima_pos[1] = x, y
    evento_raton(event, x, y, flags, param)

cv2.namedWindow(VENTANA, cv2.WINDOW_NORMAL)
cv2.resizeWindow(VENTANA, ANCHO_VENTANA, ALTO_VENTANA)
cv2.moveWindow(VENTANA, (PANT_ANCHO - ANCHO_VENTANA)//2, (PANT_ALTO - ALTO_VENTANA)//2)
cv2.setMouseCallback(VENTANA, registrar_pos)
print("Interfaz por botones. Pulsa 'q' o ESC para salir.")

while True:
    x, y = _ultima_pos
    xc_ok = x is not None and x >= ANCHO_PANEL
    canvas = vista_con_preview(x if xc_ok else None, y if xc_ok else None)
    cv2.imshow(VENTANA, np.hstack((construir_panel(), canvas)))
    tecla = cv2.waitKey(20) & 0xFF
    if tecla == ord('q') or tecla == 27: break

if video_writer is not None: video_writer.release()
cv2.destroyAllWindows()

Interfaz por botones. Pulsa 'q' o ESC para salir.
